# EDA del target

Esta notebook valida los hallazgos sobre los datos locales. Usa `competencia_01` desde el CSV generado y no reconstruye `clase_ternaria`.

La ganancia vigente es `+1.100.000` por cada `BAJA+2` correctamente contactado y `-27.500` por cualquier otro contacto.

In [1]:
from pathlib import Path
import pandas as pd

project_root = Path.cwd() if (Path.cwd() / 'data').exists() else Path.cwd().parent
target_path = project_root / 'data' / 'competencia_01.csv'
data = pd.read_csv(target_path)
data['foto_mes'] = data['foto_mes'].astype('int64')

print(f'Archivo: {target_path}')
print(f'Dimensiones: {data.shape[0]:,} filas x {data.shape[1]:,} columnas')
data.head()

Archivo: c:\Users\tomas\OneDrive\Documents\Maestria\DMEyF\dmeyf2026\data\competencia_01.csv
Dimensiones: 983,061 filas x 155 columnas


,numero_de_cliente,foto_mes,active_quarter,cliente_vip,internet,cliente_edad,cliente_antiguedad,mrentabilidad,mrentabilidad_annual,mcomisiones,...,Visa_fultimo_cierre,Visa_mpagado,Visa_mpagospesos,Visa_mpagosdolares,Visa_fechaalta,Visa_mconsumototal,Visa_cconsumos,Visa_cadelantosefectivo,Visa_mpagominimo,clase_ternaria
0,48031568,202103,1,0,0,63,106,-2002.47,-10716.53,1274.82,...,21.0,19518.72,-19589.10,0.0,3228.0,3342.19,2.0,0.0,19518.72,CONTINUA
1,48031568,202104,1,0,0,63,107,-1514.81,-14142.33,1241.08,...,23.0,19225.47,-19518.72,0.0,3258.0,3342.19,2.0,0.0,19225.47,CONTINUA
2,48031568,202105,1,0,0,63,108,-1106.31,-16421.70,1216.79,...,26.0,20961.51,-19225.47,0.0,3289.0,3342.19,2.0,0.0,17442.51,CONTINUA
3,48031568,202106,1,0,1,63,109,-956.67,-17714.29,843.35,...,21.0,5439.03,-20961.51,0.0,3319.0,3341.02,2.0,0.0,17559.81,CONTINUA
4,48031568,202107,1,0,1,63,110,-1625.57,-19282.50,425.68,...,24.0,25875.59,-5439.03,0.0,3350.0,3765.64,2.0,0.0,29841.12,NaN


## 1. Controles generales

Primero revisamos estructura, unicidad temporal y observabilidad del target.

In [2]:
tipos = pd.DataFrame({'tipo': data.dtypes.astype(str), 'nulos': data.isna().sum(), 'cardinalidad': data.nunique(dropna=True)})
duplicados = (
    data.groupby(['numero_de_cliente', 'foto_mes'], dropna=False).size()
    .rename('filas').reset_index().query('filas > 1')
)
registros_mes = data.groupby('foto_mes').size().rename('total_registros').to_frame()

print('Tipos y resumen de columnas:')
display(tipos.head(15))
print(f'Claves duplicadas cliente-mes: {len(duplicados):,}')
display(registros_mes)

Tipos y resumen de columnas:


,tipo,nulos,cardinalidad
numero_de_cliente,int64,0,169727
foto_mes,int64,0,6
active_quarter,int64,0,2
cliente_vip,int64,0,2
internet,int64,0,5
cliente_edad,int64,0,82
cliente_antiguedad,int64,0,339
mrentabilidad,float64,0,608649
mrentabilidad_annual,float64,0,915560
mcomisiones,float64,0,350570


Claves duplicadas cliente-mes: 0


,total_registros
foto_mes,
202103,162900
202104,163284
202105,163768
202106,164114
202107,164348
202108,164647


In [3]:
clases = ['CONTINUA', 'BAJA+1', 'BAJA+2']
distribucion = data.assign(clase=data['clase_ternaria'].fillna('NULL'))
distribucion = pd.crosstab(distribucion['foto_mes'], distribucion['clase']).reindex(columns=clases + ['NULL'], fill_value=0)
distribucion['total_registros'] = distribucion.sum(axis=1)
distribucion['registros_observables'] = distribucion[clases].sum(axis=1)
distribucion['porcentaje_observable'] = (100 * distribucion['registros_observables'] / distribucion['total_registros']).round(2)
display(distribucion)

validacion = pd.DataFrame(index=distribucion.index)
validacion['categorias_mas_null_igual_total'] = (distribucion[clases + ['NULL']].sum(axis=1) == distribucion['total_registros'])
validacion['porcentajes_observables_mas_null'] = (100 * distribucion[clases + ['NULL']].div(distribucion['total_registros'], axis=0).sum(axis=1)).round(6)
validacion['julio_solo_baja1_null'] = True
if 202107 in distribucion.index:
    validacion.loc[202107, 'julio_solo_baja1_null'] = distribucion.loc[202107, ['CONTINUA', 'BAJA+2']].sum() == 0
validacion['agosto_todo_null'] = True
if 202108 in distribucion.index:
    validacion.loc[202108, 'agosto_todo_null'] = distribucion.loc[202108, 'NULL'] == distribucion.loc[202108, 'total_registros']
display(validacion)

C:\Users\tomas\AppData\Local\Temp\ipykernel_19300\1569401484.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  distribucion = data.assign(clase=data['clase_ternaria'].fillna('NULL'))


clase,CONTINUA,BAJA+1,BAJA+2,NULL,total_registros,registros_observables,porcentaje_observable
foto_mes,,,,,,,
202103,160921,1019,960,0,162900,162900,100.00
202104,161181,964,1139,0,163284,163284,100.00
202105,161755,1143,870,0,163768,163768,100.00
202106,162142,874,1098,0,164114,164114,100.00
202107,0,1103,0,163245,164348,1103,0.67
202108,0,0,0,164647,164647,0,0.00


,categorias_mas_null_igual_total,porcentajes_observables_mas_null,julio_solo_baja1_null,agosto_todo_null
foto_mes,,,,
202103,True,100.0,True,True
202104,True,100.0,True,True
202105,True,100.0,True,True
202106,True,100.0,True,True
202107,True,100.0,True,True
202108,True,100.0,True,True


## 2. Calidad de datos

Medimos missing, constantes, cardinalidad, ceros y cuantiles para detectar variables poco informativas o con valores extremos.

In [4]:
missing_por_mes = data.isna().groupby(data['foto_mes']).mean().mul(100).round(2).T
missing_resumen = pd.DataFrame({'missing_total': data.isna().sum(), 'missing_pct': (100 * data.isna().mean()).round(2), 'cardinalidad': data.nunique(dropna=True)})
constantes = missing_resumen.query('cardinalidad <= 1')
casi_constantes = (100 * data.nunique(dropna=False).value_counts(normalize=True)).head(10)
numericas = data.select_dtypes(include='number').columns
ceros = pd.DataFrame({'cantidad_cero': (data[numericas] == 0).sum(), 'porcentaje_cero': (100 * (data[numericas] == 0).mean()).round(2), 'porcentaje_mayor_cero': (100 * (data[numericas] > 0).mean()).round(2)})
cuantiles = data[numericas].quantile([0, .01, .25, .5, .75, .99, 1]).T

print('Variables constantes:')
display(constantes)
print('Missing por variable, primeras 20:')
display(missing_resumen.sort_values('missing_pct', ascending=False).head(20))
print('Proporcion de ceros, primeras 20:')
display(ceros.sort_values('porcentaje_cero', ascending=False).head(20))
print('Cuantiles de variables numericas, primeras 20:')
display(cuantiles.head(20))

Variables constantes:


,missing_total,missing_pct,cardinalidad


Missing por variable, primeras 20:


,missing_total,missing_pct,cardinalidad
Master_Finiciomora,978240,99.51,127
Visa_Finiciomora,969739,98.64,142
Master_mpagospesos,579470,58.95,252626
Master_mconsumosdolares,579470,58.95,7152
Master_cconsumos,579470,58.95,110
Master_mconsumototal,579470,58.95,190197
Master_mconsumospesos,579470,58.95,190197
Master_mpagosdolares,579470,58.95,6061
Master_madelantodolares,579470,58.95,37
Master_madelantopesos,579470,58.95,127


Proporcion de ceros, primeras 20:


,cantidad_cero,porcentaje_cero,porcentaje_mayor_cero
mcuenta_corriente_adicional,982565,99.95,0.04
mpayroll2,982232,99.92,0.08
cpayroll2_trx,982232,99.92,0.08
ccheques_emitidos_rechazados,981434,99.83,0.17
mcheques_emitidos_rechazados,981434,99.83,0.17
ccheques_depositados_rechazados,981205,99.81,0.19
mcheques_depositados_rechazados,981205,99.81,0.19
cliente_vip,979949,99.68,0.32
minversion1_dolares,979829,99.67,0.33
cforex_buy,979657,99.65,0.35


Cuantiles de variables numericas, primeras 20:


,0.00,0.01,0.25,0.50,0.75,0.99,1.00
numero_de_cliente,12159854.00,1.253417e+07,25495696.00,35961151.00,49059713.00,7.459010e+07,78254956.00
foto_mes,202103.00,2.021030e+05,202104.00,202106.00,202107.00,2.021080e+05,202108.00
active_quarter,0.00,0.000000e+00,1.00,1.00,1.00,1.000000e+00,1.00
cliente_vip,0.00,0.000000e+00,0.00,0.00,0.00,0.000000e+00,1.00
internet,0.00,0.000000e+00,0.00,0.00,0.00,1.000000e+00,4.00
cliente_edad,18.00,2.400000e+01,37.00,45.00,56.00,7.900000e+01,101.00
cliente_antiguedad,1.00,6.000000e+00,63.00,128.00,190.00,3.270000e+02,380.00
mrentabilidad,-271488.39,-1.012190e+04,71.11,1416.53,3304.21,2.783189e+04,784129.03
mrentabilidad_annual,-938088.66,-6.129204e+04,1884.69,13343.47,32067.90,2.530225e+05,7023010.91
mcomisiones,-52407.90,-3.473020e+02,366.73,981.75,2181.85,1.009870e+04,513155.48


## 3. Comparacion mensual por clase

Las medias se complementan con la proporcion de clientes con valor mayor que cero, especialmente en variables con muchos ceros.

In [5]:
variables_prioritarias = [
    'ctrx_quarter', 'active_quarter', 'cpayroll_trx', 'mpayroll',
    'chomebanking_transacciones', 'mcuentas_saldo', 'Master_status',
    'Visa_status', 'cproductos', 'cliente_vip', 'ccaja_seguridad'
]
variables_disponibles = [v for v in variables_prioritarias if v in data.columns]
comparacion = data[data['clase_ternaria'].notna()].groupby(['foto_mes', 'clase_ternaria'])[variables_disponibles].agg(['count', 'mean', 'median'])
comparacion.head()

ctrx_quarter                    active_quarter  \
                               count        mean median          count   
foto_mes clase_ternaria                                                  
202103   BAJA+1                 1019   36.334642   18.0           1019   
         BAJA+2                  960   39.070833   20.0            960   
         CONTINUA             160921  119.189323  105.0         160921   
202104   BAJA+1                  964   36.405602   18.0            964   
         BAJA+2                 1139   34.291484   17.0           1139   

                                         cpayroll_trx                   \
                             mean median        count      mean median   
foto_mes clase_ternaria                                                  
202103   BAJA+1          0.808636    1.0         1019  0.060844    0.0   
         BAJA+2          0.844792    1.0          960  0.125000    0.0   
         CONTINUA        0.987696    1.0       160921  0.963771    1.0   
202104   BAJA+1          0.849585    1.0          964  0.064315    0.0   
         BAJA+2          0.829675    1.0         1139  0.160667    0.0   

                        mpayroll  ... Visa_status cproductos                   \
                           count  ...      median      count      mean median   
foto_mes clase_ternaria           ...                                           
202103   BAJA+1             1019  ...         0.0       1019  6.207066    6.0   
         BAJA+2              960  ...         0.0        960  6.432292    6.0   
         CONTINUA         160921  ...         0.0     160921  7.548766    7.0   
202104   BAJA+1              964  ...         0.0        964  6.163900    6.0   
         BAJA+2             1139  ...         0.0       1139  6.216857    6.0   

                        cliente_vip                  ccaja_seguridad  \
                              count      mean median           count   
foto_mes clase_ternaria                                                
202103   BAJA+1                1019  0.000000    0.0            1019   
         BAJA+2                 960  0.001042    0.0             960   
         CONTINUA            160921  0.002622    0.0          160921   
202104   BAJA+1                 964  0.001037    0.0             964   
         BAJA+2                1139  0.000878    0.0            1139   

                                          
                             mean median  
foto_mes clase_ternaria                   
202103   BAJA+1          0.010795    0.0  
         BAJA+2          0.016667    0.0  
         CONTINUA        0.071364    0.0  
202104   BAJA+1          0.012448    0.0  
         BAJA+2          0.011414    0.0  

[5 rows x 33 columns]

In [6]:
mayor_cero = data[data['clase_ternaria'].notna()].copy()
mayor_cero_resumen = mayor_cero.groupby(['foto_mes', 'clase_ternaria'])[variables_disponibles].apply(lambda bloque: (bloque > 0).mean().mul(100).round(2))
display(comparacion)
display(mayor_cero_resumen)

ctrx_quarter                    active_quarter  \
                               count        mean median          count   
foto_mes clase_ternaria                                                  
202103   BAJA+1                 1019   36.334642   18.0           1019   
         BAJA+2                  960   39.070833   20.0            960   
         CONTINUA             160921  119.189323  105.0         160921   
202104   BAJA+1                  964   36.405602   18.0            964   
         BAJA+2                 1139   34.291484   17.0           1139   
         CONTINUA             161181  120.488023  106.0         161181   
202105   BAJA+1                 1143   31.703412   16.0           1143   
         BAJA+2                  870   44.050575   25.0            870   
         CONTINUA             161755  120.291132  106.0         161755   
202106   BAJA+1                  874   40.556064   23.0            874   
         BAJA+2                 1098   37.897086   21.0           1098   
         CONTINUA             162142  120.790196  106.0         162142   
202107   BAJA+1                 1103   34.234814   19.0           1103   

                                         cpayroll_trx                   \
                             mean median        count      mean median   
foto_mes clase_ternaria                                                  
202103   BAJA+1          0.808636    1.0         1019  0.060844    0.0   
         BAJA+2          0.844792    1.0          960  0.125000    0.0   
         CONTINUA        0.987696    1.0       160921  0.963771    1.0   
202104   BAJA+1          0.849585    1.0          964  0.064315    0.0   
         BAJA+2          0.829675    1.0         1139  0.160667    0.0   
         CONTINUA        0.988808    1.0       161181  0.918905    1.0   
202105   BAJA+1          0.829396    1.0         1143  0.116360    0.0   
         BAJA+2          0.880460    1.0          870  0.145977    0.0   
         CONTINUA        0.988965    1.0       161755  0.985317    1.0   
202106   BAJA+1          0.872998    1.0          874  0.120137    0.0   
         BAJA+2          0.887978    1.0         1098  0.142077    0.0   
         CONTINUA        0.989016    1.0       162142  1.270041    1.0   
202107   BAJA+1          0.868540    1.0         1103  0.090662    0.0   

                        mpayroll  ... Visa_status cproductos                   \
                           count  ...      median      count      mean median   
foto_mes clase_ternaria           ...                                           
202103   BAJA+1             1019  ...         0.0       1019  6.207066    6.0   
         BAJA+2              960  ...         0.0        960  6.432292    6.0   
         CONTINUA         160921  ...         0.0     160921  7.548766    7.0   
202104   BAJA+1              964  ...         0.0        964  6.163900    6.0   
         BAJA+2             1139  ...         0.0       1139  6.216857    6.0   
         CONTINUA         161181  ...         0.0     161181  7.553818    7.0   
202105   BAJA+1             1143  ...         0.0       1143  6.062992    6.0   
         BAJA+2              870  ...         0.0        870  6.556322    7.0   
         CONTINUA         161755  ...         0.0     161755  7.550827    7.0   
202106   BAJA+1              874  ...         0.0        874  6.370709    6.0   
         BAJA+2             1098  ...         0.0       1098  6.403461    6.0   
         CONTINUA         162142  ...         0.0     162142  7.555464    7.0   
202107   BAJA+1             1103  ...         0.0       1103  6.217588    6.0   

                        cliente_vip                  ccaja_seguridad  \
                              count      mean median           count   
foto_mes clase_ternaria                                                
202103   BAJA+1                1019  0.000000    0.0            1019   
         BAJA+2                 960  0.001042    0.0             960 

ctrx_quarter  active_quarter  cpayroll_trx  mpayroll  \
foto_mes clase_ternaria                                                         
202103   BAJA+1                 86.85           80.86          4.22      4.81   
         BAJA+2                 87.40           84.48          7.60      7.60   
         CONTINUA               99.08           98.77         54.63     54.04   
202104   BAJA+1                 88.28           84.96          4.67      4.67   
         BAJA+2                 85.16           82.97          8.69      8.69   
         CONTINUA               99.13           98.88         54.72     54.12   
202105   BAJA+1                 86.09           82.94          6.04      5.95   
         BAJA+2                 91.03           88.05          7.82      7.82   
         CONTINUA               99.10           98.90         55.56     54.97   
202106   BAJA+1                 91.30           87.30          5.95      5.95   
         BAJA+2                 86.89           88.80          6.83      6.83   
         CONTINUA               99.12           98.90         55.86     55.27   
202107   BAJA+1                 86.94           86.85          4.53      4.53   

                         chomebanking_transacciones  mcuentas_saldo  \
foto_mes clase_ternaria                                               
202103   BAJA+1                               55.15           38.96   
         BAJA+2                               55.00           42.81   
         CONTINUA                             84.65           79.39   
202104   BAJA+1                               56.43           37.86   
         BAJA+2                               47.67           40.30   
         CONTINUA                             84.37           79.53   
202105   BAJA+1                               51.01           35.96   
         BAJA+2                               60.69           46.09   
         CONTINUA                             84.66           79.46   
202106   BAJA+1                               63.62           36.73   
         BAJA+2                               51.55           43.81   
         CONTINUA                             84.19           82.69   
202107   BAJA+1                               55.58           37.35   

                         Master_status  Visa_status  cproductos  cliente_vip  \
foto_mes clase_ternaria                                                        
202103   BAJA+1                   6.18         7.36       100.0         0.00   
         BAJA+2                   3.33         3.12       100.0         0.10   
         CONTINUA                 0.21         0.22       100.0         0.26   
202104   BAJA+1                   9.75        10.37       100.0         0.10   
         BAJA+2                   6.15         3.95       100.0         0.09   
         CONTINUA                 0.25         0.28       100.0         0.26   
202105   BAJA+1                   9.27        11.29       100.0         0.09   
         BAJA+2                   2.87         3.79       100.0         0.11   
         CONTINUA                 0.24         0.30       100.0         0.27   
202106   BAJA+1                   5.38         5.49       100.0         0.23   
         BAJA+2                   2.09         2.19       100.0         0.00   
         CONTINUA                 0.23         0.24       100.0         0.57   
202107   BAJA+1                   9.43        12.96       100.0         0.00   

                         ccaja_seguridad  
foto_mes clase_ternaria                   
202103   BAJA+1                     1.08  
         BAJA+2                     1.67  
         CONTINUA                   6.80  
202104   BAJA+1                     1.24  
         BAJA+2                     1.05  
         CONTINUA                   6.83  
202105   BAJA+1                     0.96  
         BAJA+2                     1.38  
         CONTINUA                   6.83  
202106   BAJA+1                     1.37  
         BAJA+2           

## 4. Bivariado: deciles y ganancia mensual

Los deciles se calculan dentro de cada mes observable. La ganancia se informa por mes y por combinacion de deciles; no se agregan meses en esta etapa.

In [7]:
GANANCIA_BAJA2 = 1_100_000
COSTO_CONTACTO = -27_500
bivariado = data[data['clase_ternaria'].notna()].copy()
bivariado['decil_ctrx_quarter'] = bivariado.groupby('foto_mes')['ctrx_quarter'].transform(lambda s: pd.qcut(s.rank(method='first'), 10, labels=False) + 1)
bivariado['decil_mcuentas_saldo'] = bivariado.groupby('foto_mes')['mcuentas_saldo'].transform(lambda s: pd.qcut(s.rank(method='first'), 10, labels=False) + 1)
bivariado['es_baja2'] = bivariado['clase_ternaria'].eq('BAJA+2')
heatmap_resumen = bivariado.groupby(['foto_mes', 'decil_ctrx_quarter', 'decil_mcuentas_saldo']).agg(clientes=('numero_de_cliente', 'size'), baja2=('es_baja2', 'sum')).reset_index()
heatmap_resumen['tasa_baja2_pct'] = (100 * heatmap_resumen['baja2'] / heatmap_resumen['clientes']).round(2)
heatmap_resumen['ganancia_contactar'] = heatmap_resumen['baja2'] * GANANCIA_BAJA2 + (heatmap_resumen['clientes'] - heatmap_resumen['baja2']) * COSTO_CONTACTO
display(heatmap_resumen.head(20))

,foto_mes,decil_ctrx_quarter,decil_mcuentas_saldo,clientes,baja2,tasa_baja2_pct,ganancia_contactar
0,202103,1,1,3083,220,7.14,163267500
1,202103,1,2,3329,173,5.20,103510000
2,202103,1,3,4116,107,2.60,7452500
3,202103,1,4,1808,31,1.71,-14767500
4,202103,1,5,1209,19,1.57,-11825000
5,202103,1,6,798,10,1.25,-10670000
6,202103,1,7,519,2,0.39,-12017500
7,202103,1,8,476,1,0.21,-11962500
8,202103,1,9,412,5,1.21,-5692500
9,202103,1,10,540,6,1.11,-8085000


In [8]:
def mostrar_heatmaps_mes(mes):
    bloque = heatmap_resumen[heatmap_resumen['foto_mes'] == mes]
    if bloque.empty:
        print(f'No hay datos observables para {mes}')
        return
    indice = ['decil_ctrx_quarter', 'decil_mcuentas_saldo']
    for metrica in ['clientes', 'tasa_baja2_pct', 'ganancia_contactar']:
        tabla = bloque.pivot(index=indice[0], columns=indice[1], values=metrica).sort_index(ascending=False)
        print(f'{mes} - {metrica}')
        display(tabla.style.background_gradient(cmap='YlOrRd').format('{:,.2f}'))

for mes in sorted(heatmap_resumen['foto_mes'].unique()):
    mostrar_heatmaps_mes(mes)

202103 - clientes


decil_mcuentas_saldo,1,2,3,4,5,6,7,8,9,10
decil_ctrx_quarter,,,,,,,,,,
10,"1,688.00",632.00,515.00,"1,129.00","1,401.00","1,549.00","1,739.00","2,086.00","2,459.00","3,092.00"
9,"1,673.00",930.00,749.00,"1,304.00","1,541.00","1,627.00","1,905.00","2,111.00","2,249.00","2,201.00"
8,"1,527.00",958.00,911.00,"1,466.00","1,550.00","1,689.00","2,039.00","2,141.00","2,073.00","1,936.00"
7,"1,445.00","1,139.00","1,031.00","1,468.00","1,588.00","1,821.00","1,990.00","2,053.00","1,955.00","1,800.00"
6,"1,345.00","1,203.00","1,157.00","1,598.00","1,706.00","1,852.00","1,950.00","1,895.00","1,901.00","1,683.00"
5,"1,279.00","1,402.00","1,426.00","1,653.00","1,751.00","1,827.00","1,861.00","1,833.00","1,694.00","1,564.00"
4,"1,334.00","1,672.00","1,607.00","1,835.00","1,859.00","1,849.00","1,712.00","1,554.00","1,474.00","1,394.00"
3,"1,415.00","2,069.00","2,114.00","1,929.00","1,870.00","1,843.00","1,470.00","1,265.00","1,158.00","1,157.00"
2,"1,501.00","2,956.00","2,664.00","2,100.00","1,815.00","1,435.00","1,105.00",876.00,915.00,923.00


202103 - tasa_baja2_pct


decil_mcuentas_saldo,1,2,3,4,5,6,7,8,9,10
decil_ctrx_quarter,,,,,,,,,,
10,0.12,0.00,0.39,0.44,0.21,0.13,0.00,0.10,0.04,0.00
9,0.12,0.22,0.13,0.15,0.06,0.12,0.05,0.19,0.13,0.14
8,0.07,0.00,0.33,0.14,0.13,0.18,0.10,0.00,0.00,0.00
7,0.21,0.44,0.39,0.14,0.06,0.33,0.05,0.05,0.00,0.00
6,0.15,0.25,0.17,0.00,0.35,0.27,0.10,0.05,0.05,0.06
5,0.16,0.43,0.28,0.36,0.46,0.16,0.11,0.05,0.06,0.06
4,0.60,0.42,0.56,0.38,0.27,0.32,0.06,0.13,0.07,0.00
3,0.57,0.48,0.80,0.47,0.53,0.33,0.07,0.16,0.17,0.00
2,0.93,1.79,1.09,0.67,0.72,0.63,0.36,0.11,0.11,0.65


202103 - ganancia_contactar


...

decil_mcuentas_saldo,1,2,3,4,5,6,7,8,9,10
decil_ctrx_quarter,,,,,,,,,,
10,2.00,2.00,nan,6.00,6.00,10.00,16.00,15.00,24.00,30.00
9,3.00,1.00,6.00,10.00,10.00,12.00,11.00,14.00,19.00,24.00
8,3.00,4.00,5.00,12.00,18.00,8.00,10.00,12.00,13.00,25.00
7,5.00,8.00,5.00,14.00,10.00,19.00,9.00,15.00,13.00,12.00
6,nan,6.00,6.00,20.00,20.00,11.00,8.00,15.00,17.00,7.00
5,5.00,7.00,8.00,18.00,17.00,13.00,13.00,16.00,8.00,6.00
4,10.00,10.00,13.00,17.00,21.00,12.00,10.00,8.00,8.00,1.00
3,24.00,16.00,22.00,7.00,7.00,11.00,11.00,4.00,5.00,3.00
2,27.00,23.00,21.00,4.00,1.00,9.00,14.00,7.00,2.00,2.00


202107 - tasa_baja2_pct


decil_mcuentas_saldo,1,2,3,4,5,6,7,8,9,10
decil_ctrx_quarter,,,,,,,,,,
10,0.00,0.00,nan,0.00,0.00,0.00,0.00,0.00,0.00,0.00
9,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
8,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
7,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
6,nan,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
5,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
4,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
3,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
2,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00


202107 - ganancia_contactar


decil_mcuentas_saldo,1,2,3,4,5,6,7,8,9,10
decil_ctrx_quarter,,,,,,,,,,
10,"-55,000.00","-55,000.00",nan,"-165,000.00","-165,000.00","-275,000.00","-440,000.00","-412,500.00","-660,000.00","-825,000.00"
9,"-82,500.00","-27,500.00","-165,000.00","-275,000.00","-275,000.00","-330,000.00","-302,500.00","-385,000.00","-522,500.00","-660,000.00"
8,"-82,500.00","-110,000.00","-137,500.00","-330,000.00","-495,000.00","-220,000.00","-275,000.00","-330,000.00","-357,500.00","-687,500.00"
7,"-137,500.00","-220,000.00","-137,500.00","-385,000.00","-275,000.00","-522,500.00","-247,500.00","-412,500.00","-357,500.00","-330,000.00"
6,nan,"-165,000.00","-165,000.00","-550,000.00","-550,000.00","-302,500.00","-220,000.00","-412,500.00","-467,500.00","-192,500.00"
5,"-137,500.00","-192,500.00","-220,000.00","-495,000.00","-467,500.00","-357,500.00","-357,500.00","-440,000.00","-220,000.00","-165,000.00"
4,"-275,000.00","-275,000.00","-357,500.00","-467,500.00","-577,500.00","-330,000.00","-275,000.00","-220,000.00","-220,000.00","-27,500.00"
3,"-660,000.00","-440,000.00","-605,000.00","-192,500.00","-192,500.00","-302,500.00","-302,500.00","-110,000.00","-137,500.00","-82,500.00"
2,"-742,500.00","-632,500.00","-577,500.00","-110,000.00","-27,500.00","-247,500.00","-385,000.00","-192,500.00","-55,000.00","-55,000.00"
